# Test du serveur MCP ProjeQtOr (Python) avec AWS Bedrock

Ce notebook lance le serveur MCP `projeqtor-mcp-server-python` en transport **stdio**, se connecte via un client MCP, expose ses tools à un modèle **Claude sur AWS Bedrock**, et fait tourner une boucle d'agent (tool use) pilotée par Bedrock.

```
Notebook ── boucle agent ──> Bedrock Converse (Claude)
   │                              │ tool_use
   └── ClientSession MCP <────────┘
         │ stdio
   serveur projeqtor-mcp-server-python
         │ httpx
   API REST ProjeQtOr
```

## Prérequis

- Python ≥ 3.11
- Le package installé : depuis `projeqtor-mcp-server-python/`, `pip install -e .`
- Accès AWS Bedrock avec un modèle Claude activé dans la région choisie (console Bedrock → *Model access*)
- Credentials AWS configurés (`aws configure`, variables d'env, ou rôle IAM)
- Une instance ProjeQtOr 12.x joignable (URL, user, password, clé API AES)

## 1. Dépendances

Le serveur MCP et `boto3` (client Bedrock). Décommenter si besoin.

In [1]:
%pip install -e .. boto3 python-dotenv
# Le ".." pointe vers projeqtor-mcp-server-python/ (parent du dossier notebooks/).

Obtaining file:///C:/Users/bourahima.coulibaly/Desktop/codebase/server-mcp-ipmp
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for projeqtor-mcp-server-python (pyproject.toml): started
  Building editable for projeqtor-mcp-server-python (pyproject.toml): finished with status 'done'
  Created wheel for projeqtor-mcp-server-python: filename=projeqtor_mcp_server_python-2.0.0-py3-none-any.whl size=4191 sha256=770c4ac213466d9d

## 2. Configuration

Renseigner les variables ProjeQtOr (passées au serveur MCP comme variables d'environnement) et Bedrock.
Créer un fichier `.env` à côté du notebook ou éditer directement ci-dessous.

> `BEDROCK_MODEL_ID` : utiliser un **inference profile** cross-region si requis par la région,
> ex. `eu.anthropic.claude-3-7-sonnet-20250219-v1:0` ou `anthropic.claude-3-5-sonnet-20241022-v2:0`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # charge un .env local si présent

# --- ProjeQtOr / iPMP (transmis au serveur MCP) ---
# Auth = HTTP Basic via PROJEQTOR_USERNAME / PROJEQTOR_PASSWORD (projeqtor
# n'accepte QUE le Basic, pas de /auth/token).
os.environ.setdefault("PROJEQTOR_BASE_URL", "https://projeqtor-beta.siinergy.net")
os.environ.setdefault("PROJEQTOR_USERNAME", "api_user")
os.environ.setdefault("PROJEQTOR_PASSWORD", "secret")
os.environ.setdefault("PROJEQTOR_API_KEY", "ma_cle_api_aes")
os.environ.setdefault("PROJEQTOR_AES_KEY_LENGTH", "128")
os.environ.setdefault("LOG_LEVEL", "WARNING")

# Garde-fous d'authentification (causes de 401 vérifiées en direct).
_user, _pwd = os.environ["PROJEQTOR_USERNAME"], os.environ["PROJEQTOR_PASSWORD"]
if _user in {"api_user", ""} or _pwd in {"secret", ""}:
    print("[!] PROJEQTOR_USERNAME/PASSWORD sont des placeholders -> 401. Renseigne un .env.")
if "@" in _user:
    print(
        f"[!] PROJEQTOR_USERNAME='{_user}' ressemble a un email SSO -> ProjeQtOr attend le LOGIN "
        "(ex. 'nom.prenom', sans @gmail.fr). 401 quasi certain."
    )

# --- AWS Bedrock ---
AWS_REGION = (
    os.environ.get("SIIGMA_BEDROCK_REGION") or os.environ.get("AWS_REGION") or "us-east-1"
)
os.environ["AWS_REGION"] = AWS_REGION
BEDROCK_MODEL_ID = (
    os.environ.get("BEDROCK_MODEL_ID") or os.environ.get("SIIGMA_AGENT_MODEL") or "zai.glm-5"
)
os.environ["BEDROCK_MODEL_ID"] = BEDROCK_MODEL_ID

# Auth Bedrock du projet = bearer token (lu automatiquement par botocore recent).
os.environ.setdefault("AWS_BEARER_TOKEN_BEDROCK", ".....")
if os.environ["AWS_BEARER_TOKEN_BEDROCK"] in {".....", ""}:
    print("[!] AWS_BEARER_TOKEN_BEDROCK non renseigne -> repli sur la chaine de creds AWS standard.")

print("Region AWS    :", AWS_REGION)
print("Modele Bedrock:", BEDROCK_MODEL_ID)
print("ProjeQtOr URL :", os.environ["PROJEQTOR_BASE_URL"])
print("ProjeQtOr user:", os.environ["PROJEQTOR_USERNAME"])


## 3. Client Bedrock

On utilise l'API **Converse** de `bedrock-runtime`, qui supporte nativement le *tool use*.

In [3]:
import boto3

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Test rapide de connectivité Bedrock (sans tools)
resp = bedrock.converse(
    modelId=BEDROCK_MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "Réponds juste 'OK Bedrock'."}]}],
    inferenceConfig={"maxTokens": 20, "temperature": 0},
)
print(resp["output"]["message"]["content"][0]["text"])

OK Bedrock


## 4. Lancer le serveur MCP en stdio + lister les tools

Le serveur est démarré comme sous-processus via `python -m projeqtor_mcp_server`.
On ouvre une `ClientSession` MCP par-dessus stdio et on récupère le catalogue de tools.

In [4]:
import os
import sys
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


SERVER_SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
assert os.path.isdir(os.path.join(SERVER_SRC, "projeqtor_mcp_server")), (
    f"src introuvable: {SERVER_SRC} — lance la cellule depuis projeqtor-mcp-server-python/notebook/"
)

server_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "projeqtor_mcp_server"],
    env={**os.environ, "MCP_TRANSPORT": "stdio", "PYTHONPATH": SERVER_SRC},
)


async def open_session(stack: AsyncExitStack) -> ClientSession:
    """Démarre le serveur MCP et retourne une ClientSession initialisée.

    Sous Jupyter/Windows, `sys.stderr` d'ipykernel n'expose pas `fileno()` ;
    or `stdio_client` le passe comme `errlog` au sous-processus -> erreur
    `UnsupportedOperation: fileno`. On fournit donc un vrai fichier comme errlog.
    Si le serveur meurt au démarrage (`McpError: Connection closed`), la cause
    exacte est écrite dans `mcp_server.log`.
    """
    errlog = stack.enter_context(open("mcp_server.log", "w", encoding="utf-8"))
    read, write = await stack.enter_async_context(stdio_client(server_params, errlog=errlog))
    session = await stack.enter_async_context(ClientSession(read, write))
    await session.initialize()
    return session


async with AsyncExitStack() as stack:
    session = await open_session(stack)
    tools = (await session.list_tools()).tools
    print(f"{len(tools)} tools exposés par le serveur MCP :\n")
    for t in tools:
        print(f"  - {t.name}: {(t.description or '').splitlines()[0][:80]}")


32 tools exposés par le serveur MCP :

  - list_projects: Lister tous les projets ProjeQtOr, avec filtres optionnels statut et type appliq
  - get_project: Récupérer le détail complet d'un projet ProjeQtOr par identifiant. Les champs de
  - create_project: Créer un projet ProjeQtOr. Les champs dans extra sont fusionnés au payload natif
  - update_project: Modifier un projet existant. Fournir id et les champs ProjeQtOr natifs à mettre 
  - get_project_kpis: Obtenir les données sources KPI d'un projet: projet, activités, travail, dépense
  - list_activities: Lister les activités d'un projet avec filtres optionnels statut et ressource res
  - create_activity: Créer une activité/tâche ProjeQtOr dans un projet. Payload chiffré AES-CTR.
  - update_activity: Modifier une activité: dates, charge, statut, responsable ou champs natifs.
  - get_gantt_data: Récupérer les données Gantt d'un projet: activités, jalons et dépendances si dis
  - add_dependency: Ajouter une dépendance entre deux activit

## 5. Conversion des tools MCP → toolConfig Bedrock

Bedrock Converse attend un `toolConfig` où chaque tool a un `inputSchema.json` (JSON Schema).
Les tools MCP exposent déjà leur `inputSchema` en JSON Schema — mapping direct.

In [5]:
def mcp_tools_to_bedrock(tools) -> dict:
    """Construit le toolConfig Bedrock à partir de la liste de tools MCP."""
    specs = []
    for t in tools:
        schema = t.inputSchema or {"type": "object", "properties": {}}
        specs.append(
            {
                "toolSpec": {
                    "name": t.name,
                    "description": (t.description or t.name)[:1000],
                    "inputSchema": {"json": schema},
                }
            }
        )
    return {"tools": specs}

## 6. Boucle d'agent Bedrock ↔ MCP

Tant que Bedrock renvoie `stopReason == "tool_use"`, on exécute le tool côté serveur MCP,
on renvoie le `toolResult`, et on relance — jusqu'à la réponse finale en texte.

In [6]:
import json

SYSTEM_PROMPT = (
    "Tu es un assistant de gestion de projet connecté à ProjeQtOr via des tools MCP. "
    "Utilise les tools pour répondre. Réponds en français, de façon concise. "
    "IMPORTANT : toutes les charges et le travail ProjeQtOr (charge, assignedWork, "
    "plannedWork, realWork, leftWork, work, marge) sont en JOURS, JAMAIS en heures. "
    "Affiche-les toujours avec l'unité 'j' (ex. '244 j'), pas 'h'."
)


def _tool_result_text(call_result) -> str:
    """Aplati le contenu d'un CallToolResult MCP en texte."""
    parts = []
    for block in call_result.content:
        parts.append(getattr(block, "text", None) or str(block))
    return "\n".join(parts) or "(vide)"


async def run_agent(session: ClientSession, tool_config: dict, question: str, max_turns: int = 8):
    """Boucle agentique : Bedrock décide, le serveur MCP exécute."""
    messages = [{"role": "user", "content": [{"text": question}]}]

    for turn in range(max_turns):
        resp = bedrock.converse(
            modelId=BEDROCK_MODEL_ID,
            system=[{"text": SYSTEM_PROMPT}],
            messages=messages,
            toolConfig=tool_config,
            inferenceConfig={"maxTokens": 1024, "temperature": 0},
        )
        out_msg = resp["output"]["message"]
        messages.append(out_msg)
        stop = resp["stopReason"]

        # Affiche le texte intermédiaire éventuel
        for block in out_msg["content"]:
            if "text" in block:
                print(f"[assistant] {block['text']}")

        if stop != "tool_use":
            return out_msg  # réponse finale

        # Exécute chaque tool demandé et renvoie les résultats
        tool_results = []
        for block in out_msg["content"]:
            if "toolUse" not in block:
                continue
            tu = block["toolUse"]
            print(f"[tool_use] {tu['name']}({json.dumps(tu['input'], ensure_ascii=False)})")
            try:
                result = await session.call_tool(tu["name"], tu["input"])
                content_text = _tool_result_text(result)
                is_error = bool(getattr(result, "isError", False))
            except Exception as exc:  # remonte l'erreur au modèle
                content_text = f"Erreur tool: {exc}"
                is_error = True
            print(f"[tool_result] {content_text[:300]}")
            tool_results.append(
                {
                    "toolResult": {
                        "toolUseId": tu["toolUseId"],
                        "content": [{"text": content_text[:8000]}],
                        "status": "error" if is_error else "success",
                    }
                }
            )
        messages.append({"role": "user", "content": tool_results})

    print("[!] max_turns atteint")
    return messages[-1]


## 7. Exécution end-to-end

Une seule cellule ouvre le serveur, construit le toolConfig, et lance l'agent sur une question.
Modifier `QUESTION` librement.

> Nécessite une instance ProjeQtOr réelle pour que les tools renvoient des données.
> Sans elle, les étapes 4 et 5 fonctionnent (catalogue de tools) mais les appels échouent au réseau.

In [ ]:
QUESTION = "Liste les projets disponibles et donne-moi un résumé du premier."

async with AsyncExitStack() as stack:
    session = await open_session(stack)
    tools = (await session.list_tools()).tools
    tool_config = mcp_tools_to_bedrock(tools)
    final = await run_agent(session, tool_config, QUESTION)

print("\n===== Réponse finale =====")
for block in final["content"]:
    if "text" in block:
        print(block["text"])

## Dépannage

- **`AccessDeniedException` / model not accessible** : activer le modèle dans la console Bedrock (*Model access*), ou utiliser un inference profile (`eu.anthropic....`) adapté à la région.
- **`ValidationException: on-demand throughput isn't supported`** : le modèle exige un inference profile → préfixer l'ID par la région (`eu.`, `us.`...).
- **Le serveur MCP ne démarre pas** : vérifier `pip install -e ..` et que `python -m projeqtor_mcp_server` fonctionne en terminal.
- **Aucune donnée ProjeQtOr** : vérifier `PROJEQTOR_BASE_URL` / credentials / clé API AES.
- **Boucle asyncio dans Jupyter** : Jupyter fournit déjà une event loop, d'où l'usage direct de `async with` / `await` dans les cellules (pas de `asyncio.run`).